# TF-IDF + LogReg

**Prepare *TherapistQA* dataset**

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load preprocessed TherapistQA from Stage 0
df = pd.read_csv('processed/therapistqa.csv')

# Split - use the same split every time for reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    df['patient_question'],
    df['dominant_distortion'],
    test_size=0.2,
    stratify=df['dominant_distortion'],
    random_state=42
)

**Prepare *PatternReframe* dataset**

In [1]:
import pandas as pd

# Load PatternReframe splits from Stage 0 preprocessing
df_train = pd.read_csv('processed/patternreframe_train.csv')
df_test = pd.read_csv('processed/patternreframe_test.csv')

X_train = df_train['thought']
y_train = df_train['pattern']
X_test = df_test['thought']
y_test = df_test['pattern']

print(f"PatternReframe train: {len(X_train)} records")
print(f"PatternReframe test: {len(X_test)} records")
print(f"Train label distribution:")
print(y_train.value_counts())
print(f"\nTest label distribution:")
print(y_test.value_counts())

PatternReframe train: 1920 records
PatternReframe test: 6807 records
Train label distribution:
pattern
Catastrophizing                                                    200
Jumping to conclusions: Fortune-telling                            199
Overgeneralization                                                 197
Jumping to conclusions: mind reading                               197
Labeling and mislabeling                                           196
Discounting the positive                                           194
Black-and-white or polarized thinking / All or nothing thinking    188
Personalization                                                    185
Mental filtering                                                   185
Should statements                                                  179
Name: count, dtype: int64

Test label distribution:
pattern
Catastrophizing                                                    726
Jumping to conclusions: Fortune-telling                 

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report

pipeline_lr = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
        stop_words='english'
    )),
    ('clf', LogisticRegression(
        C=1.0,
        class_weight='balanced',
        max_iter=5000,
        solver='lbfgs',
        multi_class='auto'
    ))
])

pipeline_lr.fit(X_train, y_train)
y_pred_lr = pipeline_lr.predict(X_test)
print(classification_report(y_test, y_pred_lr, digits=3))

                                                                 precision    recall  f1-score   support

Black-and-white or polarized thinking / All or nothing thinking      0.285     0.257     0.271       665
                                                Catastrophizing      0.465     0.434     0.449       726
                                       Discounting the positive      0.375     0.408     0.391       681
                        Jumping to conclusions: Fortune-telling      0.273     0.250     0.261       711
                           Jumping to conclusions: mind reading      0.452     0.608     0.518       699
                                       Labeling and mislabeling      0.383     0.311     0.343       669
                                               Mental filtering      0.289     0.262     0.275       660
                                             Overgeneralization      0.294     0.286     0.290       688
                                                Person

/Users/ivagoluza/Documents/cognitive-distortioins-ml/dataset_preparation/.venv/lib/python3.9/site-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
